# Netflix Content Explorer
**Artificial Intelligence Minor Project**

**Aim:** Load, clean, and analyse the Netflix titles dataset using Python, pandas, and NumPy.

**How to run:** Run each cell from top to bottom. When prompted, upload `netflix_titles.csv` downloaded from Kaggle: https://www.kaggle.com/datasets/shivamb/netflix-shows

In [1]:
# Cell 1: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Netflix Content Explorer")
print("Libraries imported successfully!")

Netflix Content Explorer
Libraries imported successfully!


In [2]:
# Cell 2: Upload the Netflix CSV file
uploaded = files.upload()
filename = next(iter(uploaded))
data = pd.read_csv(filename)

print("Dataset loaded successfully!")
print("Rows and columns:", data.shape)
display(data.head())

Saving netflix_titles.csv to netflix_titles.csv
Dataset loaded successfully!
Rows and columns: (8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
# Cell 3: Clean the dataset
for column, default in {
    'country': 'Unknown',
    'director': 'Not Listed',
    'cast': 'Not Listed',
    'rating': 'Not Rated'
}.items():
    if column in data.columns:
        data[column] = data[column].fillna(default)

# The project requirements say to drop rows missing date_added
data = data.dropna(subset=['date_added']).copy()
data['date_added'] = pd.to_datetime(data['date_added'].astype(str).str.strip(), errors='coerce')
data = data.dropna(subset=['date_added']).copy()
data['year_added'] = data['date_added'].dt.year.astype(int)

print("Data cleaning completed!")
print("Rows after cleaning:", len(data))
print("Missing values in key columns:")
print(data[['country', 'director', 'cast', 'rating', 'date_added']].isnull().sum())

Data cleaning completed!
Rows after cleaning: 8797
Missing values in key columns:
country       0
director      0
cast          0
rating        0
date_added    0
dtype: int64


In [4]:
# Cell 4: Menu-driven Netflix Content Explorer
def show_menu():
    print("\n===== NETFLIX CONTENT EXPLORER =====")
    print("1. Count of Movies vs TV Shows")
    print("2. Number of titles added each year")
    print("3. Top 10 producing countries")
    print("4. Top 5 most common genres")
    print("5. Rating distribution")
    print("6. Oldest and newest release years")
    print("7. Search titles by keyword")
    print("8. Content added in a specific year")
    print("9. Total unique countries")
    print("10. Summary statistics on release year (NumPy)")
    print("11. Quit")

def explore_netflix():
    while True:
        show_menu()
        choice = input("Enter your choice (1-11): ").strip()

        if choice == '1':
            print("\nMovies vs TV Shows:")
            print(data['type'].value_counts())

        elif choice == '2':
            print("\nTitles added each year:")
            print(data['year_added'].value_counts().sort_index())

        elif choice == '3':
            countries = data['country'].str.split(', ').explode()
            countries = countries[countries != 'Unknown']
            print("\nTop 10 producing countries:")
            print(countries.value_counts().head(10))

        elif choice == '4':
            genres = data['listed_in'].dropna().str.split(', ').explode()
            print("\nTop 5 genres:")
            print(genres.value_counts().head(5))

        elif choice == '5':
            print("\nRating distribution:")
            print(data['rating'].value_counts())

        elif choice == '6':
            years = pd.to_numeric(data['release_year'], errors='coerce').dropna()
            if len(years):
                print("Oldest release year:", int(years.min()))
                print("Newest release year:", int(years.max()))
            else:
                print("No release-year data available.")

        elif choice == '7':
            keyword = input("Enter a title keyword: ").strip()
            results = data[data['title'].str.contains(keyword, case=False, na=False)]
            if results.empty:
                print("No matching titles found.")
            else:
                print(results[['title', 'type', 'release_year', 'rating']].head(30).to_string(index=False))

        elif choice == '8':
            try:
                year = int(input("Enter the year titles were added (e.g. 2020): "))
                results = data[data['year_added'] == year]
                print(f"Number of titles added in {year}: {len(results)}")
                if not results.empty:
                    print(results[['title', 'type', 'date_added']].head(30).to_string(index=False))
            except ValueError:
                print("Please enter a valid four-digit year.")

        elif choice == '9':
            countries = data['country'].dropna().str.split(', ').explode()
            countries = countries[countries != 'Unknown']
            print("Total unique countries:", countries.nunique())

        elif choice == '10':
            years = pd.to_numeric(data['release_year'], errors='coerce').dropna().to_numpy()
            if len(years):
                print(f"Mean release year: {np.mean(years):.2f}")
                print(f"Median release year: {np.median(years):.2f}")
                print(f"Standard deviation: {np.std(years):.2f}")
                print(f"Minimum release year: {int(np.min(years))}")
                print(f"Maximum release year: {int(np.max(years))}")
            else:
                print("No release-year data available.")

        elif choice == '11' or choice == '0':
            print("Thank you for using Netflix Content Explorer!")
            break

        else:
            print("Invalid choice. Please enter a number from 1 to 11.")

explore_netflix()


===== NETFLIX CONTENT EXPLORER =====
1. Count of Movies vs TV Shows
2. Number of titles added each year
3. Top 10 producing countries
4. Top 5 most common genres
5. Rating distribution
6. Oldest and newest release years
7. Search titles by keyword
8. Content added in a specific year
9. Total unique countries
10. Summary statistics on release year (NumPy)
11. Quit
Enter your choice (1-11): 1

Movies vs TV Shows:
type
Movie      6131
TV Show    2666
Name: count, dtype: int64

===== NETFLIX CONTENT EXPLORER =====
1. Count of Movies vs TV Shows
2. Number of titles added each year
3. Top 10 producing countries
4. Top 5 most common genres
5. Rating distribution
6. Oldest and newest release years
7. Search titles by keyword
8. Content added in a specific year
9. Total unique countries
10. Summary statistics on release year (NumPy)
11. Quit
Enter your choice (1-11): 2

Titles added each year:
year_added
2008       2
2009       2
2010       1
2011      13
2012       3
2013      11
2014      24

## Three insights
After trying the menu options, write three observations from the actual output. For example, compare the number of movies and TV shows, identify the most common genre, and note the year with the most added titles. Use the values shown by your dataset rather than guessing.

In [5]:
# Optional: Save a cleaned copy of the dataset
data.to_csv('netflix_titles_cleaned.csv', index=False)
print("Cleaned dataset saved.")
files.download('netflix_titles_cleaned.csv')

Cleaned dataset saved.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>